In [5]:
!pip install faiss-cpu

from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import time

model = SentenceTransformer('all-MiniLM-L6-v2')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 82.9 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
 documents = [
    # 1
    """Machine Learning is a branch of artificial intelligence that enables computers
    to learn patterns from data and make predictions or decisions without being
    explicitly programmed for every task.""",

    # 2
    """Supervised learning uses labeled training data. The model learns a relationship
    between input features and known target outputs and can later predict outputs
    for previously unseen data.""",

    # 3
    """Unsupervised learning works with data that does not contain labeled answers.
    Common tasks include clustering, dimensionality reduction, and discovering
    hidden patterns in datasets.""",

    # 4
    """Reinforcement learning trains an agent through interaction with an environment.
    The agent receives rewards or penalties and learns a strategy that maximizes
    its long-term reward.""",

    # 5
    """Classification is a supervised machine learning task where the goal is to
    assign an input to one of several predefined categories, such as spam or
    non-spam email.""",

    # 6
    """Regression predicts continuous numerical values. Examples include predicting
    house prices, temperature, sales revenue, or the future demand for a product.""",

    # 7
    """A feature is an input variable used by a machine learning model. For example,
    in house price prediction, features may include area, number of rooms,
    location, and age of the building.""",

    # 8
    """Training data is the portion of a dataset used to teach a machine learning
    model. The model adjusts its parameters based on patterns found in this data.""",

    # 9
    """A validation dataset is used during model development to evaluate performance
    and tune hyperparameters without using the final test dataset.""",

    # 10
    """A test dataset is kept separate from training and validation data. It provides
    an unbiased estimate of how well a trained model performs on unseen examples.""",

    # 11
    """Overfitting happens when a machine learning model learns the training data
    too closely, including noise, and therefore performs poorly on new unseen data.""",

    # 12
    """Underfitting occurs when a model is too simple to capture important patterns
    in the training data. It usually performs poorly on both training and test data.""",

    # 13
    """Regularization helps reduce overfitting by adding constraints or penalties
    to a model. L1 and L2 regularization are commonly used techniques.""",

    # 14
    """Gradient descent is an optimization algorithm used to minimize a model's
    loss function. It repeatedly adjusts model parameters in the direction that
    reduces the error.""",

    # 15
    """The learning rate controls how large each parameter update is during training.
    A learning rate that is too large can cause unstable training, while one that
    is too small can make training very slow.""",

    # 16
    """A loss function measures how different a model's predictions are from the
    expected outputs. Training algorithms attempt to minimize this loss.""",

    # 17
    """Accuracy measures the percentage of predictions that a classification model
    gets correct. It can be misleading when classes are highly imbalanced.""",

    # 18
    """Precision measures how many of the examples predicted as positive are actually
    positive. It is particularly important when false positives are costly.""",

    # 19
    """Recall measures how many actual positive examples were successfully identified
    by a model. It is important when missing positive cases has serious consequences.""",

    # 20
    """The F1 score combines precision and recall into a single metric using their
    harmonic mean. It is useful when both false positives and false negatives matter.""",

    # 21
    """Natural Language Processing, or NLP, is a field of AI focused on enabling
    computers to understand, process, analyze, and generate human language.""",

    # 22
    """Tokenization breaks text into smaller units called tokens. Depending on the
    tokenizer, tokens may represent words, subwords, characters, or punctuation.""",

    # 23
    """Stop words are common words such as 'the', 'is', and 'and'. Some NLP systems
    remove them during preprocessing, although modern language models often keep
    them because context can be important.""",

    # 24
    """Stemming reduces words to simpler root-like forms by removing prefixes or
    suffixes. The resulting stem may not always be a valid dictionary word.""",

    # 25
    """Lemmatization converts words into their meaningful base or dictionary form.
    It generally uses more linguistic information than simple stemming.""",

    # 26
    """Named Entity Recognition identifies entities such as people, organizations,
    locations, dates, and products within text.""",

    # 27
    """Sentiment analysis determines the emotional or opinion-based orientation
    of text. Common categories include positive, negative, and neutral sentiment.""",

    # 28
    """Text classification assigns documents to predefined categories. Examples
    include spam detection, news classification, topic classification, and intent
    detection in chatbots.""",

    # 29
    """Text summarization creates a shorter representation of a document while
    preserving its most important information. It can be extractive or abstractive.""",

    # 30
    """Machine translation automatically converts text from one language into
    another. Modern systems commonly use neural networks and transformer models.""",

    # 31
    """Word embeddings represent words as numerical vectors. Words with similar
    meanings tend to have vectors that are close together in the embedding space.""",

    # 32
    """Sentence embeddings represent complete sentences or passages as numerical
    vectors. They are useful for semantic search, clustering, recommendation,
    and document retrieval.""",

    # 33
    """Semantic similarity measures how closely two pieces of text are related
    in meaning. Vector representations allow systems to compare texts mathematically.""",

    # 34
    """Cosine similarity compares two vectors by measuring the angle between them.
    It is widely used for comparing text embeddings because it focuses on direction
    rather than vector magnitude.""",

    # 35
    """Vector databases store numerical vector representations and provide efficient
    similarity search. They are commonly used in semantic search and retrieval
    augmented generation systems.""",

    # 36
    """FAISS stands for Facebook AI Similarity Search. It is a library designed for
    efficient similarity search and clustering of dense vectors, especially at large scale.""",

    # 37
    """A FAISS index stores vectors in a structure that allows fast nearest-neighbor
    searches. Different index types provide different trade-offs between speed,
    memory usage, and search accuracy.""",

    # 38
    """Nearest-neighbor search finds vectors that are closest to a given query vector.
    This technique is fundamental to semantic retrieval and recommendation systems.""",

    # 39
    """An embedding model converts text, images, or other information into numerical
    vectors that capture useful semantic or structural information.""",

    # 40
    """Retrieval Augmented Generation, or RAG, combines information retrieval with
    language generation. Relevant documents are retrieved first and then provided
    to a language model as context.""",

    # 41
    """A RAG system typically contains document ingestion, text chunking, embedding
    generation, vector indexing, retrieval, and response generation components.""",

    # 42
    """Chunking divides large documents into smaller pieces before generating
    embeddings. Good chunk sizes help retrieval systems find relevant information
    without providing excessive irrelevant context.""",

    # 43
    """A chatbot can use semantic retrieval to find relevant knowledge before
    generating an answer. This approach can improve responses when the required
    information is stored in an external knowledge base.""",

    # 44
    """Deep Learning is a subset of machine learning that uses neural networks
    containing multiple layers to learn complex patterns from large datasets.""",

    # 45
    """A neural network consists of interconnected computational units called neurons.
    Layers transform input data through learned weights and activation functions.""",

    # 46
    """Convolutional Neural Networks, or CNNs, are especially effective for image
    processing. Convolution layers learn spatial patterns such as edges, textures,
    and shapes.""",

    # 47
    """Recurrent Neural Networks process sequential information and were historically
    used for tasks such as language modeling, speech recognition, and time-series
    prediction.""",

    # 48
    """Transformers use attention mechanisms to process relationships between tokens.
    They have become a foundational architecture for modern NLP and large language models.""",

    # 49
    """Attention allows a model to determine which parts of an input sequence are
    more relevant when processing a particular token or generating an output.""",

    # 50
    """Large Language Models are neural networks trained on large collections of text.
    They can perform tasks such as question answering, summarization, translation,
    coding, and text generation."""
]


In [7]:
start_time = time.time()
embeddings = model.encode(documents)
generation_time = time.time() - start_time

embeddings = np.array(embeddings).astype('float32')

print("Embedding generation time:", generation_time, "seconds")
print("Embeddings shape:", embeddings.shape)
print("Embeddings dtype:", embeddings.dtype)

Embedding generation time: 1.8773691654205322 seconds
Embeddings shape: (50, 384)
Embeddings dtype: float32


In [8]:
dimension = embeddings.shape[1]  # should be 384

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Index size (number of vectors stored):", index.ntotal)

Index size (number of vectors stored): 50


In [13]:
def semantic_search(query, top_k=3):
    """
    Searches the FAISS index for the most semantically similar documents.
    Parameters:
        query (str) - the search query
        top_k (int) - number of results to return
    Returns: list of (distance, document) tuples, sorted by relevance
    """
    query_embedding = model.encode([query]).astype('float32')

    distances, indices = index.search(query_embedding, top_k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append((dist, documents[idx]))

    return results
results = semantic_search("How do computers understand language?", top_k=3)
for dist, doc in results:
     print(f"Distance: {dist:.4f}")
     print(f"Document: {doc.strip()}")
     print()

Distance: 1.1730
Document: Large Language Models are neural networks trained on large collections of text.
   They can perform tasks such as question answering, summarization, translation,
   coding, and text generation.

Distance: 1.1933
Document: Machine translation automatically converts text from one language into
   another. Modern systems commonly use neural networks and transformer models.

Distance: 1.2165
Document: Natural Language Processing, or NLP, is a field of AI focused on enabling
   computers to understand, process, analyze, and generate human language.



In [14]:
def keyword_search(query, corpus, top_k=3):
    """
    Searches documents using simple word overlap scoring.
    Parameters:
        query (str) - the search query
        corpus (list of str) - documents to search within
        top_k (int) - number of results to return
    Returns: list of (score, document) tuples, sorted by overlap count
    """
    query_words = set(query.lower().split())

    scores = []
    for doc in corpus:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words)  # count shared words
        scores.append(overlap)

    top_indices = np.argsort(scores)[::-1][:top_k]

    return [(scores[i], corpus[i]) for i in top_indices]

results = keyword_search("How do computers understand language?", documents, top_k=3)
for score, doc in results:
    print(f"Overlap count: {score}")
    print(f"Document: {doc.strip()}")
    print()

Overlap count: 1
Document: Semantic similarity measures how closely two pieces of text are related
   in meaning. Vector representations allow systems to compare texts mathematically.

Overlap count: 1
Document: Recall measures how many actual positive examples were successfully identified
   by a model. It is important when missing positive cases has serious consequences.

Overlap count: 1
Document: A loss function measures how different a model's predictions are from the
   expected outputs. Training algorithms attempt to minimize this loss.



In [15]:
queries = [
     "How do computers understand language?",
    "What is machine learning and how does it work?",
    "How can I prevent a machine learning model from overfitting?",
    "What is the difference between precision and recall?",
    "How are words converted into numerical representations?",
    "What is semantic similarity between two pieces of text?",
    "How does FAISS perform fast similarity search?",
    "What is the purpose of embeddings in NLP?",
    "How does a RAG system retrieve relevant information?",
    "Why are transformers important for modern language models?"
]

for q in queries:
    print(f"\n{'='*60}")
    print(f"QUERY: {q}")
    print(f"{'='*60}")

    print("\n--- Semantic Search (FAISS) ---")
    sem_results = semantic_search(q, top_k=3)
    for dist, doc in sem_results:
        print(f"[{dist:.4f}] {doc.strip()[:100]}...")

    print("\n--- Keyword Search ---")
    kw_results = keyword_search(q, documents, top_k=3)
    for score, doc in kw_results:
        print(f"[{score}] {doc.strip()[:100]}...")


QUERY: How do computers understand language?

--- Semantic Search (FAISS) ---
[1.1730] Large Language Models are neural networks trained on large collections of text.
   They can perform ...
[1.1933] Machine translation automatically converts text from one language into
   another. Modern systems co...
[1.2165] Natural Language Processing, or NLP, is a field of AI focused on enabling
   computers to understand...

--- Keyword Search ---
[1] Semantic similarity measures how closely two pieces of text are related
   in meaning. Vector repres...
[1] Recall measures how many actual positive examples were successfully identified
   by a model. It is ...
[1] A loss function measures how different a model's predictions are from the
   expected outputs. Train...

QUERY: What is machine learning and how does it work?

--- Semantic Search (FAISS) ---
[0.3917] Machine Learning is a branch of artificial intelligence that enables computers
   to learn patterns ...
[0.8114] Supervised learning uses